# Results NVV-Pipeline Evaluation Experiment - Full-GT and Part-GT
- yml: environment.yml
- env: nvv_isolation_pipeline
- last updated: 05.04.2026


## Setup
- Imports
- Config

In [ ]:
#Setup
import os
import sys
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from config.load_config import load_config
from config.path_factory import print_paths

from evaluation.analysis_loader import load_and_compare_workspaces

from evaluation.analysis_tables import build_rq1_capability_tables

from evaluation.analysis_tables import (
    build_setting_summary_table,
    build_best_all_screened_values_matrix,
    build_derivative_matrix,
    build_top_k_runs_table
)
from evaluation.analysis_plots import (
    plot_setting_overview,
    plot_derivative_group_comparison,
    plot_top_k_runs_per_setting,
)

cfg_path_nvs_full_gt = project_root / "experiments" / "screening" / "config_param_screening_nvs38k_full_gt.yaml"
cfg_path_nvs_part_gt = project_root / "experiments" / "screening" / "config_param_screening_nvs38k_part_gt.yaml"
cfg_path_vocal_part_gt = project_root / "experiments" / "screening" / "config_param_screening_vocal_part_gt.yaml"


config = load_config(cfg_path_nvs_full_gt)
print_paths(cfg_path_nvs_full_gt)

## Collect Pipeline Evaluation Experiment Results 
- Experiment specs for comparison


In [ ]:
# specification of full run evaluation experiments to load and compare (all settings)
specs = [
    {
        "label": "NVS-38K_EN_10_categories | full_gt",
        "cfg_path": cfg_path_nvs_full_gt,
        "dataset_name": "NVS-38K_EN_10_categories",
        "mode": "full_gt",
    },
    {
        "label": "NVS-38K_EN_10_categories | part_gt",
        "cfg_path": cfg_path_nvs_part_gt,
        "dataset_name": "NVS-38K_EN_10_categories",
        "mode": "part_gt",
    },
        {
        "label": "VOCAL_10_categories | part_gt",
        "cfg_path": cfg_path_vocal_part_gt,
        "dataset_name": "VOCAL_10_categories",
        "mode": "part_gt",
    },
]

analysis_bundle = load_and_compare_workspaces(
    specs=specs,
    param_names=[],
    param_pairs=[],
)

bundles_by_spec = analysis_bundle["bundles_by_spec"]
views_by_spec = analysis_bundle["views_by_spec"]
views = analysis_bundle["combined_views"]
setting_order = analysis_bundle["setting_order"]


In [ ]:
bundles_by_spec.keys()

## RQ1 Pipeline Capability

In [ ]:

# RQ1 Pipeline Capability
# --- collect RQ1 tables from the loaded settings ---
rq1_all = pd.concat(
    [
        bundle["results"]["rq1"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq1_tables = build_rq1_capability_tables(rq1_all)

print("RQ1 Capability Table (Full-GT):")
display(rq1_tables["full_gt"])

print("RQ1 Capability Table (Part-GT):")
display(rq1_tables["part_gt"])




In [ ]:
from evaluation.rq_plots import plot_rq1_full_gt_grouped_bars, plot_rq1_part_gt_grouped_bars

fig = plot_rq1_full_gt_grouped_bars(rq1_tables["full_gt"])
plt.show()

fig = plot_rq1_part_gt_grouped_bars(rq1_tables["part_gt"], metric="Recall", setting_order=setting_order)
plt.show()

fig = plot_rq1_part_gt_grouped_bars(rq1_tables["part_gt"], metric="Mean EOS TP", setting_order=setting_order)
plt.show()

## RQ2 Ranking Configuration: Single best

In [ ]:
from evaluation.analysis_tables import build_rq2a_single_ranking_tables

rq2a_single_all = pd.concat(
    [
        bundle["results"]["rq2a_single"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_single_tables = build_rq2a_single_ranking_tables(
    rq2a_single_all,
    top_k=10,
)

print("RQ2a Single Ranking Tables:")
for setting_name, df_table in rq2a_single_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
#toDo: check creation of columns, one of them is wrong
rq2a_single_all[[
    "setting",
    "macro_mean_dice_eos_recall",
    "macro_mean_mean_dice_eos_tp",
]].head(20)

In [ ]:

from evaluation.rq_plots import plot_rq2a_rank_vs_score

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=42,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_recall",
    top_k=42,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2a_rank_vs_score(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=42,
    setting_order=setting_order,
)
plt.show()


## RQ2 Configuration Combination: Selected Set

In [ ]:
from evaluation.analysis_tables import build_rq2a_selected_set_tables

rq2a_selected_all = pd.concat(
    [
        bundle["results"]["rq2a_selected_set"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq2a_selected_tables = build_rq2a_selected_set_tables(
    rq2a_single_all,
    rq2a_selected_all,
)

print("RQ2a Selected Set Tables:")
for setting_name, df_table in rq2a_selected_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

## RQ2 Audio Derivative Aggregation

In [ ]:
# RQ2 Audio Derivative Aggregation - Just for inspection, not for final presentation. ToDo: update artifact with insertion rate and remove precision and f1 for part_gt
# Keep derivative matrices separated by metric / mode
derivative_matrix_f1 = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1", #toDo: also recall and dice
)

derivative_matrix_recall = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall", #toDo also eos and eos_tp and insertion_rate
)

print("\nDerivative Matrix (F1, full_gt):")
display(derivative_matrix_f1)
print("\nDerivative Matrix (Recall, part_gt):")
display(derivative_matrix_recall)



In [ ]:
# RQ2b Audio Derivative Aggregation - Group Comparison (Thesis Tables)
from evaluation.analysis_tables import build_rq2b_derivative_tables
## RQ2b Audio Derivative Aggregation

rq2b_tables = build_rq2b_derivative_tables(views["derivative_comparison"])

print("RQ2b Audio Derivative Table (Full-GT):")
display(rq2b_tables["full_gt"])

print("RQ2b Audio Derivative Table (Part-GT):")
display(rq2b_tables["part_gt"])

In [ ]:
from evaluation.rq_plots import plot_rq2b_boxplot_with_points
fig = plot_rq2b_boxplot_with_points(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2b_boxplot_with_points(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

## RQ2b Inspection of top 2

In [ ]:
from evaluation.analysis_tables import inspect_top_n_rq2a_configs_by_group

df_top2_derivatives_full = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="full_gt",
    group_by="audio_derivative_group",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_derivatives_full)

df_top2_derivatives_part = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="part_gt",
    group_by="audio_derivative_group",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_derivatives_part)

## Configuration VAD Mask Inspection

In [ ]:
from evaluation.analysis_tables import build_rq2b_vad_mask_tables

rq2b_vad_tables = build_rq2b_vad_mask_tables(
    rq2a_single_all,
    setting_order=setting_order,
)

display(rq2b_vad_tables["full_gt"])
display(rq2b_vad_tables["part_gt"])

In [ ]:
from evaluation.rq_plots import plot_rq2b_vad_mask_boxplot_with_points

fig = plot_rq2b_vad_mask_boxplot_with_points(
    rq2a_single_all,
    mode="full_gt",
    score_col="macro_mean_f1",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

fig = plot_rq2b_vad_mask_boxplot_with_points(
    rq2a_single_all,
    mode="part_gt",
    score_col="macro_mean_recall",
    top_k=None,
    setting_order=setting_order,
)
plt.show()

In [ ]:
df_top2_vad_full = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="full_gt",
    group_by="vad_mask",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_vad_full)

df_top2_vad_part = inspect_top_n_rq2a_configs_by_group(
    rq2a_single_all,
    mode="part_gt",
    group_by="vad_mask",
    top_n=2,
    setting_order=setting_order,
)
display(df_top2_vad_part)

## RQ3 NVV Coverage

In [ ]:
## RQ3 NVV Coverage

from evaluation.analysis_tables import (
    build_rq3_full_gt_label_tables,
    build_rq3_part_gt_event_tables,
    build_rq3_global_tables,
)

rq3_label_all = pd.concat(
    [
        bundle["results"]["rq3_label"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq3_global_all = pd.concat(
    [
        bundle["results"]["rq3_global"]
        for bundle in analysis_bundle["bundles_by_spec"].values()
    ],
    ignore_index=True,
)

rq3_full_gt_label_tables = build_rq3_full_gt_label_tables(rq3_label_all)
rq3_part_gt_event_tables = build_rq3_part_gt_event_tables(rq3_label_all)
rq3_global_tables = build_rq3_global_tables(rq3_global_all)



## RQ3 Global Metrics Tables - Full-GT and Part-GT

In [ ]:
display(rq3_global_tables["full_gt"])
display(rq3_global_tables["part_gt"])

## RQ3 Label Coverage (Full-GT) Table and Plot

In [ ]:
print("RQ3 Event Tables (Full-GT):")
for setting_name, df_table in rq3_full_gt_label_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
from evaluation.rq_plots import plot_rq3_label_coverage

fig = plot_rq3_label_coverage(
    rq3_full_gt_label_tables["NVS-38K_EN_10_categories | full_gt"],
    setting=None,
)
plt.show()


## RQ3 Label quality (full-GT)

In [ ]:
from evaluation.rq_plots import plot_rq3_label_quality

fig = plot_rq3_label_quality(
    rq3_full_gt_label_tables["NVS-38K_EN_10_categories | full_gt"]
)
plt.show()

## RQ3 Global Recall Comparison (Full-GT - Part-GT)

In [ ]:
from evaluation.rq_plots import plot_rq3_global_recall_comparison

fig = plot_rq3_global_recall_comparison(
    rq3_global_all,
    setting_order=setting_order,
)
plt.show()

## RQ3 Event Samples (Part-GT) Tables

In [ ]:
## RQ3 Event Samples (Part-GT) Tables
print("RQ3 Event Tables (Part-GT):")
for setting_name, df_table in rq3_part_gt_event_tables.items():
    print(f"\n=== {setting_name} ===")
    display(df_table)

In [ ]:
display(rq1_tables["full_gt"].columns)
display(rq2a_single_all.columns)
display(rq3_global_tables["full_gt"].columns)